In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
import math

np.random.seed(11)
n = 100

In [15]:
theta_true = 42
def inv_pareto(u, th):
    return (1 - u)**(1 / (1 - th))

sample = inv_pareto(stats.uniform(loc=0, scale=1).rvs(size=n), th=theta_true)
print(f"Выборка: {sample}")

Выборка: [1.00165225 1.02106905 1.02881083 1.02070006 1.02283363 1.02257835
 1.05846001 1.01864026 1.04937822 1.01063717 1.0037032  1.00613756
 1.0391975  1.00638503 1.03677648 1.05851028 1.01935845 1.00377729
 1.02411116 1.01170764 1.06004711 1.00619317 1.00653221 1.01619009
 1.01673206 1.00508524 1.01150325 1.02727097 1.01562967 1.00150301
 1.01317554 1.01319676 1.01458068 1.00058867 1.0078402  1.07601897
 1.05372622 1.07133946 1.03530408 1.01397844 1.0699648  1.00876058
 1.02218077 1.02182898 1.01427571 1.00705337 1.00983333 1.025435
 1.0081822  1.01041846 1.01071837 1.00962542 1.04394675 1.01943487
 1.00045231 1.03355467 1.01715763 1.03749334 1.01678153 1.00912131
 1.0455979  1.00465391 1.0079857  1.01627579 1.04909883 1.00299
 1.01013714 1.01870449 1.02805185 1.00271462 1.03492924 1.02005224
 1.01791561 1.01176232 1.00925775 1.05542985 1.13188937 1.00470941
 1.00581254 1.02718528 1.03590547 1.04313392 1.00882345 1.00573796
 1.00183918 1.01158211 1.02036667 1.05152861 1.00081991 1.

In [16]:
beta = 0.95

In [17]:
def med(th):
    return 2 ** (1/(th-1))

In [27]:
theta = 1 + 1 / (np.mean(np.log(sample)))
med_true = med(theta_true)
med_hat = med(theta)
z_low = stats.norm.ppf((1 - beta)/2)
z_high = stats.norm.ppf((1 +beta)/2)
ci_asym_med_left = med_hat - med_hat*np.log(2)/(np.sqrt(n)*(theta-1)) * z_high
ci_asym_med_right = med_hat - med_hat*np.log(2)/(np.sqrt(n)*(theta-1)) * z_low
ci_asym_med_len = ci_asym_med_right - ci_asym_med_left
print(f"Медиана: {med_true:.4f}, Оценка медианы: {med_hat:.4f}")
print(f"Асимптотический доверительный интервал для медианы: {ci_asym_med_left:.4f} < θ < {ci_asym_med_right:.4f}")
print(f"Длина интервала: {ci_asym_med_len:.4f}")

Медиана: 1.0170, Оценка медианы: 1.0147
Асимптотический доверительный интервал для медианы: 1.0118 < θ < 1.0176
Длина интервала: 0.0058


In [29]:
z_low = stats.norm.ppf((1-beta)/2)
z_high = stats.norm.ppf((1+beta)/2)
ci_asym_left = theta - z_high / (np.sum(np.log(sample))/np.sqrt(n))
ci_asym_right = theta - z_low / (np.sum(np.log(sample))/np.sqrt(n))
ci_asym_len = ci_asym_right - ci_asym_left
print(f"θ = {theta_true},  Оценка θ: {theta}")
print(f"Асимптотический доверительный интервал для θ: {ci_asym_left:.4f} < θ < {ci_asym_right:.4f}")
print(f"Длина интервала: {ci_asym_len:.4f}") 

θ = 42,  Оценка θ: 48.57061581084614
Асимптотический доверительный интервал для θ: 39.2469 < θ < 57.8943
Длина интервала: 18.6473


In [ ]:
B = 1000

def boot_np(samp, B):
    diffs = []
    m = len(samp)
    for _ in range(B):
        samp_b = np.random.choice(samp, size=m, replace=True)        
        diffs += [(1 + 1 / (np.mean(np.log(samp_b)))) - theta]
    return sorted(np.array(diffs))

diffs_np = boot_np(sample, B)
idsample_low = int((1 - beta)/2 * B - 1)
idsample_high = int((1 + beta)/2 * B - 1)

ci_np_left = theta - diffs_np[idsample_high]
ci_np_right = theta - diffs_np[idsample_low]
ci_np_len = ci_np_right - ci_np_left
print(f"Непараметрический бутстраповский доверительный интервал для θ (ОМП):", ci_np_left, "< θ <", ci_np_right)
print(f"Длина интервала: {ci_np_len:.4f}")

Непараметрический бутстраповский доверительный интервал для θ (ОМП): 38.700605754041796 < θ < 56.065377667694655
Длина интервала: 17.36477191365286


In [ ]:
B = 50000

def boot_p(samp, B):
    diffs = []
    m = len(samp)
    for _ in range(B):
        samp_b = inv_pareto(stats.uniform(loc=0, scale=1).rvs(size=m), th=theta)        
        diffs += [(1 + 1 / (np.mean(np.log(samp_b)))) - theta]
    return sorted(np.array(diffs))

diffs_p = boot_p(sample, B)
idsample_low = int((1 - beta)/2 * B - 1)
idsample_high = int((1 + beta)/2 * B - 1)
ci_p_left = theta - diffs_p[idsample_high]
ci_p_right = theta - diffs_p[idsample_low]
ci_p_len = ci_p_right - ci_p_left
print(f"Параметрический бутстраповский доверительный интервал для theta (ОМП):", ci_p_left, "< theta <", ci_p_right)
print(f"Длина интервала: {ci_p_len:.4f}")

Параметрический бутстраповский доверительный интервал для theta(ОМП): 37.66146697881158 < theta < 56.689147236435815
length = 19.027680257624233
